# Overhead Lines and Underground Cables


All overhead and underground AC lines are represented as `ACLineSegment` objects. Like all `ConductingEquipment`, all lines and cables are defined with two Terminal objects with `ACDCTerminal:sequenceNumber` set to 1 and 2 to represent the two ends of the line, rather than specifying a from-bus and to-bus. There are four different ways to specify `ACLineSegment` impedances and admittances. The first two use positive and zero sequence values; the third specifies the lower triangular R, X, and B values for each conductor phase; the fourth uses the conductor material, geometry, and spacing. In all cases, the `Conductor:length` attribute is required. A combination of all four methods may be used in a single model to define the network. 



The first way, depicted in the figure below, is to specify the individual positive sequence and zero sequence R, X, and B values as `ACLineSegment` attributes, in a manner similar to the method used to define line impedance in many bus-branch transmission analysis tools, such as PSSE. A power system application importing the CIM model would directly use the specified attributes to run a power flow solution. The `PerLengthImpedance` attribute is left as null. The classes and attributes used in this method are shown in the figure below.


In [1]:
from cimgraph import utils
from mermaid import Mermaid
import cimgraph.data_profile.cimhub_2023 as cim

In [2]:
diagram_text = utils.get_mermaid([cim.ACLineSegment,cim.Conductor])
Mermaid(diagram_text)


The second way, displayed in the figure below, is to specify the positive and zero sequence impedance and admittance values on a per-unit-length basis as attributes of the `PerLengthSequenceImpedance` class. A power system application importing the CIM model would multiply the specified R, X, and B values by the `Conductor:length` to determine the overall line impedance. When using this method, all the `ACLineSegment` attributes should be null.


In [3]:
diagram_text = utils.get_mermaid([cim.ACLineSegment,cim.Conductor,cim.PerLengthImpedance,cim.PerLengthSequenceImpedance])
Mermaid(diagram_text)


The third way to specify line parameters, shown in the figure below, is to define R, X, and B values for each conductor phase. This method is more useful for distribution networks for which an unbalanced power flow solution is needed. The impedance and admittance values are specified as attributes of the `PhaseImpedanceData` class, which inherits from `PerLengthPhaseImpedance`. Again, all the ACLineSegment attributes are left null. Only conductorCount from 1 to 3 is supported, and there will be 1, 3 or 6 reverse-associated `PhaseImpedanceData` instances that define the lower triangle of the Z and Y matrices per unit length. The row and column attributes must agree with `ACLineSegmentPhase:sequenceNumber`.


In [4]:
diagram_text = utils.get_mermaid([cim.ACLineSegment,cim.Conductor,cim.PerLengthImpedance,cim.PerLengthSequenceImpedance,cim.ACLineSegmentPhase,cim.SinglePhaseKind,cim.PhaseImpedanceData,cim.PerLengthPhaseImpedance])
Mermaid(diagram_text)

<p style="text-align: justify;">
The fourth way, shown in the figure below, is to specify wire/cable geometry and spacing data instead of impedances. Unlike the previous three methods (where the impedance values were specified through the 61970 Wires package), all attributes of the line are specified through physical attributes as part of the 61968 AssetInfo package.  Conductor spacing is specified by association to *WireSpacingInfo* and *WirePosition*. Conductor geometry is specified through the attributes of *WireInfo*. Cables are specified through the CableInfo class and associated *ConcentricNeutralCableInfo* and *TapeShieldCableInfo* classes.
</p>


If there are *ACLineSegmentPhase* instances reverse-associated to the *ACLineSegment*, then per-phase modeling applies. There are several use cases for the *ACLineSegmentPhase* class:
1)	single-phase, two-phase, or three-phase unbalanced primary lines
2)	low-voltage secondary lines using phases s1 and s2
3)	associated WireInfo data where the WireSpacingInfo association exists
4)	assign specific phases to the matrix rows and columns in PerLengthPhaseImpedance. 


<p style="text-align: justify;">
It is the application’s responsibility to propagate phasing through terminals to other components, and to identify any miswiring. It is also the application’s responsibility to calculate the impedance and admittance values for the electrical network from the conductor geometry and spacing. The associations between all the AssetInfo classes described above are shown in the figure below.
</p>

In [5]:
diagram_text = utils.get_mermaid([cim.ACLineSegment,cim.Conductor,cim.PerLengthImpedance,cim.PerLengthSequenceImpedance,cim.ACLineSegmentPhase,cim.SinglePhaseKind,cim.PhaseImpedanceData,cim.PerLengthPhaseImpedance,cim.PerLengthLineParameter,cim.WireAssemblyInfo,cim.WireSpacingInfo,cim.WireInfo,cim.ConcentricNeutralCableInfo,cim.TapeShieldCableInfo,cim.CableInfo,cim.WireMaterialKind,cim.CableConstructionKind,cim.CableShieldMaterialKind,cim.WireInsulationKind,cim.CableOuterJacketKind])
Mermaid(diagram_text)

----

Some examples related to overhead lines and underground cables are discussed below.


In [6]:
import os
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel
import cimgraph.data_profile.cimhub_2023 as cim
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'

In [7]:
# Basic line examples use the standard IEEE 13 model
network = FeederModel(connection=XMLFile(filename='../sample_models/ieee13.xml'), container=None)

# Wire, cable, and spacing (AssetInfo) examples use the model variant with asset data
assets = FeederModel(connection=XMLFile(filename='../sample_models/ieee13_assets.xml'), container=None)

Example 1: What are the phases of Line named 632645?


In [8]:
name = '632645'

# find_by_attribute returns all ACLineSegments whose name contains the search string
line = network.find_by_attribute(cim.ACLineSegment, 'name', name)[0]

result = [str(phase.phase) for phase in line.ACLineSegmentPhases]

print(result)

['SinglePhaseKind.C', 'SinglePhaseKind.B']


In [9]:
diagram_text = utils.get_mermaid_path(line,['ACLineSegmentPhases','[0]','phase'])
diagram_text = utils.add_mermaid_path(line,'ACLineSegmentPhases[1].phase', diagram_text)
Mermaid(diagram_text)

Example 2: Find the length of the line named 632645?

In [10]:
name = '632645'

line = network.find_by_attribute(cim.ACLineSegment, 'name', name)[0]

print(line.length)

152.4


In [11]:
diagram_text = utils.get_mermaid_path(line,'length')
Mermaid(diagram_text)

Example 3: What is the nominal voltage of the line named 632645?

In [12]:
name = '632645'

line = network.find_by_attribute(cim.ACLineSegment, 'name', name)[0]
base_voltage = line.BaseVoltage

print(base_voltage.nominalVoltage)

4160.0


In [13]:
diagram_text = utils.get_mermaid_path(line,'BaseVoltage')
diagram_text = utils.add_mermaid_path(base_voltage, 'nominalVoltage', diagram_text)
Mermaid(diagram_text)

In [14]:
# diagram_text = utils.get_mermaid_path(cim.ACLineSegment,'BaseVoltage')
# Mermaid(diagram_text)

Example 4: Find how many lines use overhead wire acsr_4/0?

In [15]:
name = 'acsr_4/0'
lines = set()

# WireInfo objects are only present in the model that carries AssetInfo data
for wire_info in assets.find_by_attribute(cim.OverheadWireInfo, 'name', name):
    for phase in wire_info.ACLineSegmentPhases:
        lines.add(phase.ACLineSegment.mRID)

print(len(lines))

5


Example 5: Find the names of buses connected to line named 632645?

In [16]:
name = '632645'

line = network.find_by_attribute(cim.ACLineSegment, 'name', name)[0]
results = [terminal.ConnectivityNode.name for terminal in line.Terminals]

print(results)

['632', '645']


Example 6: Find the wire position data of phase conductor wires used by line 650632?

In [17]:
name = '650632'
results = []

line = assets.find_by_attribute(cim.ACLineSegment, 'name', name)[0]
if line.WireSpacingInfo is not None:
    for wire_position in line.WireSpacingInfo.WirePositions:
        results.append({'sequence': wire_position.sequenceNumber,
                        'x': wire_position.xCoord,
                        'y': wire_position.yCoord})

print(results)

[{'sequence': 1, 'x': -1.2192, 'y': 8.5344}, {'sequence': 2, 'x': -0.3048, 'y': 8.5344}, {'sequence': 3, 'x': 0.9144, 'y': 8.5344}, {'sequence': 4, 'x': 0.0, 'y': 7.3152}]


Example 7: The thermal rating of cables in the model?

In [18]:
results = []

# Cables are described by ConcentricNeutralCableInfo and TapeShieldCableInfo
for cable_class in [cim.ConcentricNeutralCableInfo, cim.TapeShieldCableInfo]:
    for cable_info in assets.list_by_class(cable_class):
        results.append({'name': cable_info.name,
                        'thermal rating': cable_info.ratedCurrent})

print(results)

[{'name': 'cn_250', 'thermal rating': 260.0}, {'name': 'ts_1/0', 'thermal rating': 165.0}]


Example 8: Identify which lines in the model are underground cables?

In [19]:
results = set()

for line in assets.list_by_class(cim.ACLineSegment):
    for phase in line.ACLineSegmentPhases:
        # Underground cables use the concentric-neutral or tape-shield WireInfo types
        if isinstance(phase.WireInfo, (cim.ConcentricNeutralCableInfo, cim.TapeShieldCableInfo)):
            results.add(line.name)

print(results)

{'692675', '684652'}


Example 9: Find the names of lines in the model that are underground cables using insulation made from cross linked polyethylene?

In [20]:
material = cim.WireInsulationKind.crosslinkedPolyethylene
results = []

# Both cable types are searched for the matching insulation material
for cable_class in [cim.TapeShieldCableInfo, cim.ConcentricNeutralCableInfo]:
    for wire_info in assets.find_by_attribute(cable_class, 'insulationMaterial', material):
        for phase in wire_info.ACLineSegmentPhases:
            if phase.ACLineSegment is not None:
                results.append(phase.ACLineSegment.name)

print(results)

['684652', '692675', '692675', '692675']


Example 10: What is the radius of the wire used in the phases of line with mRID 6DEF3353-8276-402F-AC8E-3DEF4A396FFE?

In [21]:
results = []
line = assets.get_object(mRID='6DEF3353-8276-402F-AC8E-3DEF4A396FFE')

# Each phase conductor carries its own WireInfo with physical geometry
for phase in line.ACLineSegmentPhases:
    if phase.WireInfo is not None:
        results.append({'phase': str(phase.phase),
                        'coreRadius': phase.WireInfo.coreRadius,
                        'radius': phase.WireInfo.radius,
                        'gmr': phase.WireInfo.gmr})

print(results)

[{'phase': 'SinglePhaseKind.A', 'coreRadius': 0.0, 'radius': 0.0117729, 'gmr': 0.00947928}, {'phase': 'SinglePhaseKind.B', 'coreRadius': 0.0, 'radius': 0.0117729, 'gmr': 0.00947928}, {'phase': 'SinglePhaseKind.C', 'coreRadius': 0.0, 'radius': 0.0117729, 'gmr': 0.00947928}, {'phase': 'SinglePhaseKind.N', 'coreRadius': 0.0, 'radius': 0.0071501, 'gmr': 0.002481072}]


Example 11: What is the resistance of line 645646?

In [22]:
name = '645646'
results = []

for line in network.find_by_attribute(cim.ACLineSegment, 'name', name):
    # Method 1: positive/zero sequence impedance stored directly on the line
    if line.r is not None:
        results.append({'r': line.r, 'r0': line.r0})

    per_length_impedance = line.PerLengthImpedance

    # Method 2: per-length sequence impedance
    if isinstance(per_length_impedance, cim.PerLengthSequenceImpedance):
        results.append({'per_length_r': per_length_impedance.r,
                        'per_length_r0': per_length_impedance.r0})

    # Method 3: per-length phase impedance matrix
    if isinstance(per_length_impedance, cim.PerLengthPhaseImpedance):
        for phase_impedance_data in per_length_impedance.PhaseImpedanceData:
            results.append({'phase_impedance_r': phase_impedance_data.r})

print(results)

[{'phase_impedance_r': 0.00082257118}, {'phase_impedance_r': 0.00012837529}, {'phase_impedance_r': 0.00082605086}]


Example 12: What are the location xy coordinates of line named 632645?

In [23]:
name = '632645'
results = []

for line in network.find_by_attribute(cim.ACLineSegment, 'name', name):
    for position in line.Location.PositionPoints:
        results.append(position.xPosition)
        results.append(position.yPosition)

print(results)

['200', '250', '100', '250']
